In [ ]:
# Cell: Rasterize BoPs shapefile by BType1 (1=hard, 2=mixed, 3=soft)

import geopandas as gpd
import rasterio
from rasterio import features
from rasterio.crs import CRS
from rasterio.transform import from_bounds
import numpy as np
import os
from pathlib import Path
import pandas as pd              

# ────────────────────────────────────────────────
# USER INPUTS ─────────────────────────────────────
# ────────────────────────────────────────────────
shp_dir_or_path = r"path\to\a\shp\file"  # ← Change this! for instace: "C:\Users\username\...\BoPs_WCVI.shp"
#   - Can be a folder containing .shp (it will find .shp), or direct path to .shp file

output_raster_path = r"path\to\the\output\tif\file"           # ← Change this! for instace: "C:\Users\username\...\BoPs_WCVI_10m.tif"

resolution = 10.0          # meters (10 m to match Sentinel-2; 20 m also common)
nodata_value = 0           # Value for areas outside polygons 
field_name = "BType1"      # Attribute field to rasterize (case-sensitive!)

# ────────────────────────────────────────────────
# Find the shapefile ──────────────────────────────
# ────────────────────────────────────────────────
if os.path.isdir(shp_dir_or_path):
    shp_path = None
    for ext in [".shp", ".SHP"]:
        candidates = list(Path(shp_dir_or_path).glob(f"*{ext}"))
        if len(candidates) == 1:
            shp_path = candidates[0]
            break
        elif len(candidates) > 1:
            print("Multiple .shp files found! Please specify exact path.")
            raise ValueError("Ambiguous shapefile")
    if shp_path is None:
        raise FileNotFoundError(f"No .shp file found in {shp_dir_or_path}")
else:
    shp_path = Path(shp_dir_or_path)
    if not shp_path.exists() or not shp_path.suffix.lower() == ".shp":
        raise FileNotFoundError(f"Shapefile not found or not .shp: {shp_path}")

print(f"Reading shapefile: {shp_path}")

# ────────────────────────────────────────────────
# Load vector data ────────────────────────────────
# ────────────────────────────────────────────────
gdf = gpd.read_file(shp_path)

if field_name not in gdf.columns:
    raise ValueError(f"Field '{field_name}' not found in shapefile. Available: {list(gdf.columns)}")

# Ensure BType1 is numeric (int)
gdf[field_name] = pd.to_numeric(gdf[field_name], errors='coerce').astype('Int64')

# Drop rows with missing BType1 (or handle as needed)
gdf = gdf.dropna(subset=[field_name]).copy()

print(f"Loaded {len(gdf)} polygons. Unique BType1 values: {gdf[field_name].unique()}")

# ────────────────────────────────────────────────
# Prepare raster metadata ─────────────────────────
# ────────────────────────────────────────────────
bounds = gdf.total_bounds  # minx, miny, maxx, maxy
width = int(np.ceil((bounds[2] - bounds[0]) / resolution))
height = int(np.ceil((bounds[3] - bounds[1]) / resolution))

transform = from_bounds(
    bounds[0], bounds[1], bounds[2], bounds[3],
    width, height
)

profile = {
    'driver': 'GTiff',
    'height': height,
    'width': width,
    'count': 1,
    'dtype': 'int16',         
    'crs': gdf.crs,
    'transform': transform,
    'nodata': nodata_value,
    'compress': 'lzw',         # optional: smaller file
}

# ────────────────────────────────────────────────
# Rasterize ───────────────────────────────────────
# ────────────────────────────────────────────────
print("Rasterizing...")

# Create shapes list: (geometry, value) pairs
shapes = [(geom, value) for geom, value in zip(gdf.geometry, gdf[field_name])]

# Burn into numpy array
raster_array = features.rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    all_touched=True,          # include all pixels touched by polygon
    fill=nodata_value,
    dtype='int16'
)

# ────────────────────────────────────────────────
# Write to GeoTIFF ────────────────────────────────
# ────────────────────────────────────────────────
with rasterio.open(output_raster_path, 'w', **profile) as dst:
    dst.write(raster_array, 1)

print(f"Raster saved to: {output_raster_path}")
print(f"Resolution: {resolution} m | CRS: {gdf.crs} | Shape: {raster_array.shape}")
print("Done!")

Reading shapefile: C:\Users\mohsenghanbari\OneDrive - University of Victoria\Desktop\substrateBoPs\BOPsToOpenDataWCVI\BoPs_WCVI.shp
Loaded 110313 polygons. Unique BType1 values: <IntegerArray>
[3, 2, 1]
Length: 3, dtype: Int64
Rasterizing...
Raster saved to: C:\Users\mohsenghanbari\OneDrive - University of Victoria\Desktop\substrateBoPs\BOPsToOpenDataWCVI\BoPs_WCVI_10m.tif
Resolution: 10.0 m | CRS: EPSG:3005 | Shape: (29137, 37678)
Done!
